# Chapter 4 — Tasks, State and Actions

*The task is not a prompt. The state is not the chat history. The action is not a free-form dict.*

## Objective

Take the Chapter 2 loop and refit it with typed objects: `TaskSpec`, `AgentState`, `Action`. Same behavior, but every step is now serializable and replayable.

In [ ]:
from glassloop.core import (
    Action, ActionKind, AgentState, AskUser, BaseAgent, Escalate, Finish, StepRecord,
    TaskSpec, ToolCall, ValidationRule, parse_action, run_loop,
)
from pydantic import ValidationError

## TaskSpec carries more than a prompt

A task has a goal, inputs, expected outputs, constraints and validation rules. The goal cannot be empty.

In [ ]:
task = TaskSpec(
    goal='Classify and summarize a customer complaint',
    inputs={'message': 'I was charged a $35 overdraft fee.'},
    expected_outputs=['category', 'summary'],
    constraints=['Do not invent facts', 'Do not expose PII'],
    validation=[ValidationRule(name='schema', description='must match output schema')],
)
print(task.model_dump_json(indent=2))

In [ ]:
try:
    TaskSpec(goal='   ')
except ValidationError as e:
    print('rejected empty goal:', e.errors()[0]['msg'])

## AgentState round-trips losslessly

State is `to_dict` / `from_dict`. The serialized form is the unit of audit, replay and persistence.

In [ ]:
state = AgentState(task=task, step=2, messages=[{'role': 'user', 'content': 'hi'}])
d = state.to_dict()
back = AgentState.from_dict(d)
print('round-trip equal:', back == state)

## Actions are a discriminated union

`ToolCall`, `AskUser`, `Finish`, `Escalate`. The `kind` field is constrained by `Literal`, so a wrong kind fails at construction time, not silently downstream.

In [ ]:
actions = [
    ToolCall(tool_name='search', arguments={'q': 'overdraft'}),
    AskUser(question='Did you authorize this transaction?'),
    Finish(output={'category': 'complaint'}),
    Escalate(reason='UDAAP risk', context={'flags': ['fee', 'overdraft']}),
]
for a in actions:
    print(f'{a.kind:<10} {type(a).__name__}')

try:
    ToolCall(kind='ask_user', tool_name='x')
except ValidationError as e:
    print('wrong kind rejected:', e.errors()[0]['msg'])

## Audit logs deserialize via `parse_action`

Given a JSON-shaped dict, `parse_action` dispatches on `kind` and returns the correct subclass. This is how the Chapter 12 audit log replays actions.

In [ ]:
for a in actions:
    rebuilt = parse_action(a.model_dump())
    assert type(rebuilt) is type(a)
print('all actions round-trip through parse_action')

## Refit the Chapter 2 agent

Same loop, same agent, but every step is now serializable. Inspect a step record's `state_after.to_dict()` and you have an audit-ready record.

In [ ]:
class GreedyAgent(BaseAgent):
    def __init__(self, target): self.target = target
    def propose_action(self, state):
        if state.step >= self.target:
            return Finish(output={'pings': state.step})
        return ToolCall(tool_name='ping', arguments={})

class PingEnv:
    def step(self, action): return {'pong': True}

records = list(run_loop(GreedyAgent(2), PingEnv(), AgentState(task=task), max_steps=10))
for r in records:
    print(r.step, r.action.kind, r.state_after.status)

## Anti-patterns flagged here

- Tasks expressed only as a prompt string.
- State stored in module globals.
- `action` as a free-form dict — no schema, no audit, no replay.

In [ ]:
# Self-check: state and actions round-trip; the final action is Finish.
assert AgentState.from_dict(records[-1].state_after.to_dict()) == records[-1].state_after
assert records[-1].action.kind == 'finish'
print('OK')